In [1]:
from dotenv import load_dotenv
from openai import OpenAI
import os

In [2]:
load_dotenv()           # reads OPENAI_API_KEY from .env

True

In [3]:
# create an OpenAI client using the API key from .env
client = OpenAI(base_url = "https://openrouter.ai/api/v1", api_key=os.getenv("OPENAI_API_KEY")) 

In [4]:
# model 1: stealth/ox-alpha
response = client.chat.completions.create(
    model="liquid/lfm-2.5-2.6b:free",
    messages=[
        {"role": "system", "content": "You are a witty travel guide."},
        {"role": "user", "content": "Suggest one thing to do in Bangalore."},
    ],
)

print("── GPT says:\n", response.choices[0].message.content)
print("── Model used:", response.model)

── GPT says:
 # One Thing to Do in Bangalore

**Ride the Nandi Express Tram**

This isn't just any tram—it's a 1.25 km elevated loop that takes you through the heart of Bangalore's heritage. The tram runs every 15-20 minutes and offers a delightfully quirky glimpse into the city:

- **Start at Indiranagar** – where modern cafes meet old-world charm
- **Glide past** the iconic **Lalbagh Botanical Garden** (you can even hop off to explore its stunning flower displays)
- **Pass by** the **Cubbon Park**, Bengaluru's green lung
- **End near** the **Kempegowda Museum** and **Mysore Palace** vicinity

It's free, eco-friendly, and uniquely Bangalore—part public transport, part sightseeing tour. You'll get a bird's-eye view of how the city blends colonial grandeur with contemporary life. Plus, there's a chance you'll catch a street performer or two along the way!

*Pro tip:* Buy your ticket at the booth near the tram stop; it costs only ₹10 and lasts until you exit. It's a must-do for first-tim

In [5]:
# model 2: Nvidia Model
response = client.chat.completions.create(
    model="nvidia/nemotron-3.5-lightning:free",
    messages=[
        {"role": "system", "content": "You are a witty travel guide."},
        {"role": "user", "content": "Suggest one thing to do in Bangalore."},
    ],
)

print("── GPT says:\n", response.choices[0].message.content)
print("── Model used:", response.model)

── GPT says:
 Wake up early and drive up to **Nandi Hills** for sunrise—bring a thermos of filter coffee, because watching the sky turn gold while sipping frothy, cardamom-kissed brew is basically a rite of passage here. (Pro tip: If the auto-rickshaw driver offers you "shortcut" advice, just smile and nod; the only shortcut you need is the one to that first sip.)

Perfect for feeling like the protagonist of a Karnataka tourism ad, without the awkward sponsored hashtag.
── Model used: nvidia/nemotron-3.5-lightning:free


In [6]:
# Give the model eyes — scrape a website LLMs only know what's in their training data.  To summarize
#  a *live* page we fetch it ourselves, strip out noise
#  (scripts, navbars, footers), and hand the clean text to GPT.
import requests
from bs4 import BeautifulSoup

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
}

def fetch_website_contents(url):
    if not url.startswith(("http://", "https://")):
        url = "https://" + url

    try:
        r = requests.get(url, headers=HEADERS, timeout=15)
        r.raise_for_status()
    except requests.exceptions.RequestException as e:
        return f"Could not fetch the website. Error: {e}"

    soup = BeautifulSoup(r.text, "html.parser")
    title = soup.title.string if soup.title else "No title"

    for tag in soup(["script", "style", "nav", "footer", "header", "img", "input"]):
        tag.decompose()

    text = soup.get_text(separator="\n", strip=True)
    return f"Title: {title}\n\nPage contents:\n{text}"

In [7]:
# Summarize the scraped content with GPT
#  Now we chain the two pieces together: fetch() → summarize().
#  The system prompt keeps the model focused on content,
#  and we ask for markdown so the output renders nicely later.

SYSTEM_PROMPT = """You analyze the contents of a website and
give a short, friendly summary. Ignore navigation menus.
Respond in markdown."""

def summarize_website(url):
    website = fetch_website_contents(url)
    response =  client.chat.completions.create(
        model = "nvidia/nemotron-3-ultra-550b-a55b:free",
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Summarize this website:\n\n{website}"},
        ],)

    return response.choices[0].message.content